# 28 - Human-Agent Collaboration

## Scenario: Human-in-the-Loop (HITL) Escalation

A fully autonomous agent is dangerous. The **Human-in-the-Loop (HITL)** pattern allows the agent to execute safely up to a certain boundary, and then explicitly PAUSE and ask a human for approval before executing a critical action.

In this notebook, we simulate a Northstar agent that requires Manager Approval before issuing refunds over $100.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. The Pause and Approve Workflow

In [2]:
def execute_refund(amount: int):
    print(f"  💸 [Bank Tool] Processing refund for ${amount}...")

def agent_refund_flow(requested_amount: int):
    print(f"\n🧠 [Agent] User requested a refund of ${requested_amount}.")
    
    if requested_amount > 100:
        print(f"🚦 [Agent] Amount exceeds $100 threshold. Escalating to Human Manager.")
        
        # In a real app, the execution state is saved to a DB (like LangGraph checkpoints)
        # and the script stops. A webhook wakes it back up when the manager clicks "Approve".
        print("  ⏸️ Agent execution paused. Waiting for approval...")
        
        # We simulate the manager's response
        manager_approved = True
        
        if manager_approved:
            print("  ✅ [Human] Manager Approved.")
            execute_refund(requested_amount)
        else:
            print("  ❌ [Human] Manager Denied.")
    else:
        print("✅ [Agent] Amount is within safe limits. Auto-executing.")
        execute_refund(requested_amount)

# 1. Safe Auto-execution
agent_refund_flow(50)

# 2. Requires HITL
agent_refund_flow(250)



🧠 [Agent] User requested a refund of $50.
✅ [Agent] Amount is within safe limits. Auto-executing.
  💸 [Bank Tool] Processing refund for $50...

🧠 [Agent] User requested a refund of $250.
🚦 [Agent] Amount exceeds $100 threshold. Escalating to Human Manager.
  ⏸️ Agent execution paused. Waiting for approval...
  ✅ [Human] Manager Approved.
  💸 [Bank Tool] Processing refund for $250...


## Checkpoint

**1. What is the primary purpose of Human-in-the-Loop (HITL)?**
- A) To make the agent slower.
- B) To provide a safety boundary where an agent can automate the investigative work but explicitly pause to require human authorization before executing high-risk, irreversible actions.
- C) To teach the LLM to code.
- D) To bypass the token budget.
